In [ ]:
from typing import Any, SupportsFloat
import pygame
import random
import numpy as np
import time
import keyboard
import copy
import numpy
import gymnasium as gym
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv, VecEnv
from gymnasium.wrappers import TimeLimit
import cv2
from gymnasium.envs.registration import register


class SnakeEnv:

    def __init__(self, height, width) -> None:
        self.height = height
        self.width = width
        self.observationSpace = [
            [0 for _ in range(self.width)] for _ in range(self.height)]
        pass

    def getValue(self, y, x):
        return self.observationSpace[y][x]

    def setValue(self, y, x, value):
        if self.observationSpace[y][x] != value:
            self.observationSpace[y][x] = value

    def setNull(self, y, x):
        self.observationSpace[y][x] = 0

    def print(self):
        print()
        for y in range(len(self.observationSpace)):
            print(self.observationSpace[y])

    def getDesignationValue(self, value):
        result = []
        for y in range(len(self.observationSpace)):
            for x in range(len(self.observationSpace[y])):
                if self.observationSpace[y][x] == value:
                    result.append((y, x))
        return result

    def getEnvironmentalData(self):
        return self.observationSpace


import random
from collections import deque

class Wall:
    def __init__(self, snakeEnv: SnakeEnv, num_interior_walls: int = 3) -> None:
        self.value = 1
        self.snakeEnv = snakeEnv
        self.walls = []
        self.num_interior_walls = num_interior_walls
        self.generate()

    def generate(self):
        snakeEnv = self.snakeEnv
        # Always place border walls
        for y in range(snakeEnv.height):
            for x in range(snakeEnv.width):
                if x == 0 or x == snakeEnv.width - 1 or \
                        y == 0 or y == snakeEnv.height - 1:
                    snakeEnv.setValue(y, x, self.value)
                    self.walls.append((y, x))

        # Try to place interior walls randomly
        placed = 0
        attempts = 0
        max_attempts = self.num_interior_walls * 20  # avoid infinite loops

        while placed < self.num_interior_walls and attempts < max_attempts:
            attempts += 1
            # Pick a random interior cell
            y = random.randint(1, snakeEnv.height - 2)
            x = random.randint(1, snakeEnv.width - 2)

            if self.isWall(y, x):
                continue

            # Tentatively add wall and check safety
            self.walls.append((y, x))
            if self._is_safe():
                snakeEnv.setValue(y, x, self.value)
                placed += 1
            else:
                # Reject — would create a deadlock cell
                self.walls.pop()

    def _is_safe(self) -> bool:
        """
        Check two conditions:
        1. All non-wall cells are still connected (no isolated regions).
        2. No non-wall cell has fewer than 2 non-wall neighbours
           (a cell with only 1 free neighbour is a dead-end that
           prevents a Hamiltonian path on the interior grid).
        """
        snakeEnv = self.snakeEnv
        wall_set = set(self.walls)

        free_cells = [
            (y, x)
            for y in range(snakeEnv.height)
            for x in range(snakeEnv.width)
            if (y, x) not in wall_set
        ]

        if not free_cells:
            return False

        # connectivity via BFS
        visited = set()
        queue = deque([free_cells[0]])
        visited.add(free_cells[0])

        while queue:
            cy, cx = queue.popleft()
            for dy, dx in [(-1,0),(1,0),(0,-1),(0,1)]:
                nb = (cy + dy, cx + dx)
                if nb not in visited and nb not in wall_set \
                        and 0 <= nb[0] < snakeEnv.height \
                        and 0 <= nb[1] < snakeEnv.width:
                    visited.add(nb)
                    queue.append(nb)

        if len(visited) != len(free_cells):
            return False  # disconnected region

        #no dead-end cells
        for (cy, cx) in free_cells:
            free_neighbours = sum(
                1 for dy, dx in [(-1,0),(1,0),(0,-1),(0,1)]
                if (cy + dy, cx + dx) not in wall_set
                and 0 <= cy + dy < snakeEnv.height
                and 0 <= cx + dx < snakeEnv.width
            )
            if free_neighbours < 2:
                return False  # dead-end cell -> Hamiltonian path impossible

        return True

    def isWall(self, y, x):
        return (y, x) in set(self.walls)



    def reset(self):
        snakeEnv = self.snakeEnv

        # Clear only interior walls from the grid
        for (y, x) in self.walls:
            if not self._is_border(y, x):
                snakeEnv.setValue(y, x, 0)

        # Keep only border walls
        self.walls = [(y, x) for (y, x) in self.walls if self._is_border(y, x)]

        # Re-place border walls on the fresh grid
        for (y, x) in self.walls:
            snakeEnv.setValue(y, x, self.value)

        # Re-run interior placement
        placed = 0
        attempts = 0
        max_attempts = self.num_interior_walls * 20

        while placed < self.num_interior_walls and attempts < max_attempts:
            attempts += 1
            y = random.randint(1, snakeEnv.height - 2)
            x = random.randint(1, snakeEnv.width - 2)

            if self.isWall(y, x):
                continue

            self.walls.append((y, x))
            if self._is_safe():
                snakeEnv.setValue(y, x, self.value)
                placed += 1
            else:
                self.walls.pop()

    def _is_border(self, y, x) -> bool:
        snakeEnv = self.snakeEnv
        return x == 0 or x == snakeEnv.width - 1 or \
            y == 0 or y == snakeEnv.height - 1


class Food:

    def __init__(self, snakeEnv: SnakeEnv, seed=None) -> None:
        self.value = 2
        self.snakeEnv = snakeEnv
        if seed == None:
            seed = int(time.time())
        random.seed(seed)
        self.foodCoordinates = ()
        self.generate()
        pass

    def generate(self, remove=False):
        if remove and len(self.foodCoordinates) == 2:
            y, x = self.foodCoordinates
            self.snakeEnv.setNull(y, x)
        emptyPosition = self.snakeEnv.getDesignationValue(0)
        lenEmptyPosition = len(emptyPosition)
        if lenEmptyPosition <= 0:
            return True
        if lenEmptyPosition == 1:
            y, x = emptyPosition[0]
        else:
            ri = random.randint(0, len(emptyPosition) - 1)
            y, x = emptyPosition[ri]
        self.foodCoordinates = (y, x)
        self.snakeEnv.setValue(y, x, self.value)
        return False

    def getFoodCoordinates(self):
        return self.foodCoordinates

    def isFood(self, y, x):
        vy, vx = self.foodCoordinates
        return vy == y and vx == x

    def rendering(self):
        if len(self.foodCoordinates) == 2:
            y, x = self.foodCoordinates
            self.snakeEnv.setValue(y, x, self.value)


class SnakeBody:

    UP = 1
    DOWN = 2
    LEFT = 3
    RIGHT = 4

    def __init__(self, snakeEnv: SnakeEnv) -> None:
        self.snakeEnv = snakeEnv
        self.head_value = 3
        self.body_value = 4
        # Length
        self.len = 1
        # Direction of movement
        self.movingDirection = 1
        self.body = [(int(snakeEnv.height / 2), int(snakeEnv.width / 2))]
        self.end = ()
        self.generate()
        pass

    def getBody(self):
        return self.body

    def getHead(self):
        return self.body[0]

    def length(self):
        return len(self.body)

    def setAction(self, action):
        if action == 0:
            return
        if (action % 2 == 0 and action - 1 == self.movingDirection) or \
                (action % 1 == 0 and action + 1 == self.movingDirection):
            return
        self.movingDirection = action

    def lengthen(self):
        if len(self.end) == 2:
            y, x = self.end
            self.body.append((y, x))
            self.snakeEnv.setValue(y, x, self.body_value)
            self.end = ()

    def move(self):
        y, x = self.body[0]
        if self.movingDirection == SnakeBody.UP:
            y -= 1
        elif self.movingDirection == SnakeBody.DOWN:
            y += 1
        elif self.movingDirection == SnakeBody.LEFT:
            x -= 1
        elif self.movingDirection == SnakeBody.RIGHT:
            x += 1
        else:
            raise RuntimeError(f"Exception move : {self.movingDirection}")
        self.body.insert(0, (y, x))
        ey, ex = self.body.pop()
        self.end = (ey, ex)
        self.snakeEnv.setNull(ey, ex)
        self.generate()
        return (y, x)

    def headBodyStack(self):
        hy, hx = self.body[0]
        for i in range(1, len(self.body)):
            y, x = self.body[i]
            if hy == y and hx == x:
                return True
        return False

    def isBody(self, hy, hx):
        for i in range(1, len(self.body)):
            y, x = self.body[i]
            if hy == y and hx == x:
                return True
        return False

    def generate(self):
        y, x = self.body[0]
        self.snakeEnv.setValue(y, x, self.head_value)
        for i in range(1, len(self.body)):
            y, x = self.body[i]
            self.snakeEnv.setValue(y, x, self.body_value)


class TcsV2Env(gym.Env):

    metadata = {"render_modes": ["human", "rgb_array"], "render_fps": 4}

    def __init__(self, **kwargs) -> None:
        self.image_size = kwargs.get("image_size", 10)
        self.show = False
        self.height = kwargs.get("height", 50)
        self.width = kwargs.get("width", 50)
        self.windowcolor = [(0, 0, 0), (255, 255, 255), (255, 0, 0),
                            (0, 0, 255), None]
        self.WINDOW_WIDTH = self.width * self.image_size
        self.WINDOW_HEIGHT = self.height * self.image_size
        self.WINDOW_BLACK = (0, 0, 0)
        self.obs_type = kwargs.get("obs_type", "rgb")
        self.outputType = kwargs.get("outputType", "Discrete")
        if self.outputType == "Discrete":
            self.action_space = gym.spaces.Discrete(5)
        elif self.outputType == "Box":
            self.action_space = gym.spaces.Box(
                low=0, high=4, shape=(1,), dtype=np.uint8
            )
        if self.obs_type == "rgb":
            self.observation_space = gym.spaces.Box(
                low=0, high=255, shape=(self.height * self.image_size, self.width * self.image_size, 3), dtype=np.uint8)
        if self.obs_type == "data":
            self.observation_space = gym.spaces.Box(
                low=0, high=255, shape=(8,), dtype=np.float32)
        self.render_mode = kwargs.get("render_mode", None)
        self.mp4 = kwargs.get("mp4", False)
        self.mp4fps = kwargs.get("mp4fps", 30)
        assert self.render_mode is None or self.render_mode in self.metadata["render_modes"]
        super().__init__()

    def get_action_meanings(self):
        return ["NOOP", "UP", "DOWN", "LEFT", "RIGHT"]

    def step(self, action: Any) -> tuple[Any, SupportsFloat, bool, bool, dict[str, Any]]:
        if isinstance(action, np.ndarray):
            action = action[0]
            action = round(action)
        info = {}
        reward = 0
        self.snakebody.setAction(action)
        y, x = self.snakebody.move()
        done = False
        if self.snakebody.isBody(y, x):
            done = True
            info["message"] = "Come into contact with the body"
        if self.wall.isWall(y, x):
            done = True
            info["message"] = "hit the wall"
        if self.food.isFood(y, x):
            self.snakebody.lengthen()
            if self.food.generate():
                done = True
                vn_step = 1000 - self.num_step
                if vn_step < 0:
                    vn_step = 0
                reward += 100 * (vn_step / 1000)
                info["message"] = "Damn, the screen is completely covered with it"
                if self.show:
                    time.sleep(3)
            reward += 1
            # self.distanceFood = 20
        # else:
        #     fy, fx = self.food.getFoodCoordinates()
        #     jy = abs(fy - y)
        #     jx = abs(fx - x)
        #     distance = jy + jx
        #     if distance < self.distanceFood:
        #         self.distanceFood = distance
        #         reward += 1 - ((1 / 20) * distance)
        if reward < 0:
            reward = 0


        self.score += reward
        observation = self.environmentalData()

        if self.mp4 and self.obs_type != "rgb":
            self.render()

        self.num_step += 1
        return observation, reward, done, False, info

    def reset(self, *, seed: int | None = None, options: dict[str, Any] | None = None) -> tuple[Any, dict[str, Any]]:
        self.snakeEnv = SnakeEnv(self.height, self.width)

        if hasattr(self, 'wall'):
            self.wall.snakeEnv = self.snakeEnv  # point wall to the fresh grid
            self.wall.reset()
        else:
            self.wall = Wall(self.snakeEnv)

        self.snakebody = SnakeBody(self.snakeEnv)
        self.food = Food(self.snakeEnv, seed=seed)
        observation = self.environmentalData()
        self.score = 0
        self.num_step = 0
        # self.distanceFood = 20
        return observation, {}


    # Changed observations, added danger observations
    # new observation space 225 * 16 * 4 * 36 = 518_400 rows
    def environmentalData(self):
        if self.obs_type == "rgb":
            return self.render()
        elif self.obs_type == "data":
            hy, hx = self.snakebody.getHead()
            fy, fx = self.food.getFoodCoordinates()

            # Relative food position (more useful than absolute)
            food_dy = fy - hy
            food_dx = fx - hx

            # Danger flags in each direction
            danger_up = int(self.wall.isWall(hy-1, hx) or self.snakebody.isBody(hy-1, hx))
            danger_down = int(self.wall.isWall(hy+1, hx) or self.snakebody.isBody(hy+1, hx))
            danger_left = int(self.wall.isWall(hy, hx-1) or self.snakebody.isBody(hy, hx-1))
            danger_right = int(self.wall.isWall(hy, hx+1) or self.snakebody.isBody(hy, hx+1))

            # Current direction (1–4)
            direction = self.snakebody.movingDirection

            # Snake length (normalised)
            length = self.snakebody.length()

            return np.array([food_dy, food_dx,
                 danger_up, danger_down, danger_left, danger_right,
                 direction,
                 length], dtype=np.float32)

    def render(self):
        if (self.render_mode == "human") and ("window" not in self.__dict__ or self.window == None):
            pygame.init()
            self.window = pygame.display.set_mode(
                (self.width * 10, self.height * 10))
            pygame.display.set_caption("tcs")
            self.window.fill(self.WINDOW_BLACK)
            self.show = True
        # Create a canvas
        if "canvas" not in self.__dict__ or self.canvas == None:
            self.canvas = pygame.Surface(
                (self.WINDOW_WIDTH, self.WINDOW_HEIGHT))
        if self.show and ("canvasN" not in self.__dict__ or self.canvasN == None):
            self.canvasN = pygame.Surface(
                (self.width * 10, self.height * 10)
            )
        self.canvas.fill(self.WINDOW_BLACK)
        if self.show:
            self.canvasN.fill(self.WINDOW_BLACK)

        observation = self.snakeEnv.getEnvironmentalData()
        for y in range(len(observation)):
            for x in range(len(observation[y])):
                value = observation[y][x]
                if value == 0 or self.windowcolor[value] == None:
                    continue
                pygame.draw.rect(
                    self.canvas, self.windowcolor[value], (x * self.image_size, y * self.image_size, self.image_size, self.image_size))
                if self.show:
                    pygame.draw.rect(
                        self.canvasN, self.windowcolor[value], (
                            x * 10, y * 10, 10, 10)
                    )
        body = self.snakebody.getBody()
        bodylen = len(body)
        if bodylen > 1:
            colosNm = round(105 / (bodylen - 1))
            dcolosNm = 150 + colosNm
            for i in range(bodylen - 2, 0, -1):
                y, x = body[i]
                vvColos = (0, dcolosNm, 0)
                pygame.draw.rect(
                    self.canvas, vvColos, (x * self.image_size, y * self.image_size, self.image_size, self.image_size))
                if self.show:
                    pygame.draw.rect(
                        self.canvasN, vvColos, (
                            x * 10, y * 10, 10, 10)
                    )
                if dcolosNm >= 255:
                    continue
                dcolosNm += colosNm
                if dcolosNm > 255:
                    dcolosNm = 255
            y, x = body[-1]
            vbcolos = (232, 255, 1)
            pygame.draw.rect(
                self.canvas, vbcolos, (x * self.image_size, y * self.image_size, self.image_size, self.image_size))
            if self.show:
                pygame.draw.rect(
                    self.canvasN, vbcolos, (
                        x * 10, y * 10, 10, 10)
                )

        result = np.transpose(
            np.array(pygame.surfarray.pixels3d(self.canvas)), axes=(1, 0, 2)
        )
        if self.render_mode == "human":
            show_image = np.transpose(
                np.array(pygame.surfarray.pixels3d(self.canvasN)), axes=(1, 0, 2)
            )
            self.window.blit(self.canvasN, self.canvasN.get_rect())
            # Cleanup Event
            pygame.event.pump()
            pygame.display.update()
            # pygame.display.flip()
            # event = pygame.event.poll()
            # # pygame.display.update()
            # if event.type == pygame.QUIT:
            #     pygame.quit()
            #     exit()
            if self.mp4:
                if "outMp4" not in self.__dict__ or self.outMp4 == None:
                    height, width, _ = show_image.shape
                    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
                    self.outMp4 = cv2.VideoWriter(
                        "1.mp4", fourcc, self.mp4fps, (width, height))
                self.outMp4.write(show_image)


        return result

    def close(self):
        del self.snakebody
        del self.wall
        del self.food
        del self.snakeEnv
        del self.score
        if self.mp4 and hasattr(self, "outMp4") and self.outMp4 is not None:
            self.outMp4.release()
        return super().close()


class ConstructEnv:

    def __init__(self) -> None:
        gym.register(id="tcs-v2", entry_point=TcsV2Env, max_episode_steps=2000,
                     kwargs={})
        pass

    def getDummyVecEnvs(self, num=32, **kwargs) -> DummyVecEnv:
        return DummyVecEnv([lambda: self.getEnv(**kwargs) for _ in range(num)])

    def getSudoVecEnvs(self, num=6, **kwargs) -> SubprocVecEnv:
        return SubprocVecEnv([lambda: self.getEnv(**kwargs) for _ in range(num)])

    def getEnv(self, **kwargs) -> TimeLimit:
        return TimeLimit(gym.make("tcs-v2", **kwargs), max_episode_steps=5000)


register(
    id="tcs-v2", entry_point=TcsV2Env, max_episode_steps=2000,
                     kwargs={}
)

In [ ]:
import torch
import torch.nn as nn

#Apperently RND treat a food respawns as a novelty and do not encourage agent to explore, but passively get reward
# in new random position target(obs) changes because pixels changed, which gives high error
# and snake learns that "chase novelty" = chase food respawns, not plan paths

class RNDNetwork(nn.Module):
    def __init__(self, obs_shape, output_dim=256):
        super().__init__()
        c, h, w = obs_shape[2], obs_shape[0], obs_shape[1]

        self.net = nn.Sequential(
            nn.Conv2d(c, 32, kernel_size=3, stride=2),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=1),
            nn.ReLU(),
            nn.Flatten(),
        )
        dummy = torch.zeros(1, c, h, w)
        cnn_out = self.net(dummy).shape[1]

        self.head = nn.Sequential(
            nn.Linear(cnn_out, 256),
            nn.ReLU(),
            nn.Linear(256, output_dim)
        )


    def forward(self, x):
        if x.ndim == 3:
            x = x.unsqueeze(0)
        x = x.permute(0, 3, 1, 2).float() / 255.0
        return self.head(self.net(x))


class RND:
    def __init__(self, obs_shape, output_dim=256, lr=1e-4, beta=0.5, device="cpu"):
        self.device = device
        self.beta = beta

        self.target = RNDNetwork(obs_shape, output_dim).to(device)
        self.predict = RNDNetwork(obs_shape, output_dim).to(device)


        # freeze target
        # init target with large scale
        for m in self.target.modules():
            if isinstance(m, (nn.Linear, nn.Conv2d)):
                nn.init.orthogonal_(m.weight, gain=2) #calibrated to get loss ~1 on the first step
                nn.init.constant_(m.bias, 0)

        # init predictor with small scale
        for m in self.predict.modules():
            if isinstance(m, (nn.Linear, nn.Conv2d)):
                nn.init.orthogonal_(m.weight, gain=0.01)
                nn.init.constant_(m.bias, 0)

        self.optimizer = torch.optim.Adam(self.predict.parameters(), lr=lr)

    def intrinsic_reward(self, obs):
        obs_t = torch.tensor(obs, dtype=torch.float32).to(self.device)
        with torch.no_grad():
            target_feat = self.target(obs_t)
        predict_feat = self.predict(obs_t)
        error = nn.functional.mse_loss(predict_feat, target_feat, reduction="none")
        return error.mean(dim=-1).detach().cpu().numpy()  # shape (n_envs,)

    def update(self, obs):
        obs_t = torch.tensor(obs, dtype=torch.float32).to(self.device)
        with torch.no_grad():
            target_feat = self.target(obs_t)
        predict_feat = self.predict(obs_t)
        loss = nn.functional.mse_loss(predict_feat, target_feat)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        return loss.item()


In [ ]:
# So ICM may cope with it cause encoder only cares about action-caused changes
# food respawning without the snake moving assigns low inverse model error
# so snake must actually explore to get bonus

class ICMModule(nn.Module):
    def __init__(self, obs_shape, n_actions, feature_dim=256):
        super().__init__()
        c, h, w = obs_shape[2], obs_shape[0], obs_shape[1]

        # shared encoder — same structure as ActorCritic CNN
        self.encoder = nn.Sequential(
            nn.Conv2d(c, 32, kernel_size=3, stride=2),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=1),
            nn.ReLU(),
            nn.Flatten(),
        )
        dummy = torch.zeros(1, c, h, w)
        cnn_out = self.encoder(dummy).shape[1]
        self.head = nn.Sequential(
            nn.Linear(cnn_out, feature_dim),
            nn.ReLU()
        )

        # inverse model: (phi_t, phi_t+1) → action
        self.inverse = nn.Sequential(
            nn.Linear(feature_dim * 2, 256),
            nn.ReLU(),
            nn.Linear(256, n_actions)
        )

        # forward model: (phi_t, action_onehot) → phi_t+1
        self.forward_model = nn.Sequential(
            nn.Linear(feature_dim + n_actions, 256),
            nn.ReLU(),
            nn.Linear(256, feature_dim)
        )

        self.n_actions = n_actions
        self.feature_dim = feature_dim

    def _encode(self, obs):
        if obs.ndim == 3:
            obs = obs.unsqueeze(0)
        obs = obs.permute(0, 3, 1, 2).float() / 255.0
        return self.head(self.encoder(obs))

    def forward(self, obs, next_obs, actions):
        phi = self._encode(obs) # (B, feature_dim)
        phi_next = self._encode(next_obs)   # (B, feature_dim)

        # inverse loss — predicts action from state transition
        pred_actions = self.inverse(torch.cat([phi, phi_next], dim=-1))
        inverse_loss = nn.functional.cross_entropy(pred_actions, actions)

        # forward loss — predicts next feature from current + action
        actions_onehot = nn.functional.one_hot(
            actions, self.n_actions).float()
        pred_phi_next = self.forward_model(
            torch.cat([phi, actions_onehot], dim=-1))

        # forward error per sample = intrinsic reward
        forward_error = 0.5 * (pred_phi_next - phi_next.detach()
                               ).pow(2).mean(dim=-1)

        return forward_error, inverse_loss, phi_next.detach()


class ICM:
    def __init__(self, obs_shape, n_actions, n_envs,
                 feature_dim=256, lr=1e-4, beta=0.5,
                 icm_lambda=0.2, device="cpu"):
        self.device = device
        self.beta = beta
        self.n_envs = n_envs

        self.icm = ICMModule(obs_shape, n_actions, feature_dim).to(device)
        self.optimizer = torch.optim.Adam(self.icm.parameters(), lr=lr)

        # running normalisation
        self.error_mean = 0.0
        self.error_var = 1.0
        self.error_count = 1e-8
        self.icm_lambda = icm_lambda

    def _update_stats(self, errors):
        n = len(errors)
        total = self.error_count + n
        delta = np.mean(errors) - self.error_mean
        self.error_mean += delta * n / total
        self.error_var = (
            self.error_var * self.error_count +
            np.var(errors) * n +
            delta**2 * self.error_count * n / total
        ) / total
        self.error_count = total

    def compute_intrinsic(self, obs, next_obs, actions):
        obs_t = torch.tensor(obs, dtype=torch.float32).to(self.device)
        next_obs_t = torch.tensor(next_obs, dtype=torch.float32).to(self.device)
        actions_t = torch.tensor(actions, dtype=torch.long).to(self.device)

        forward_errors, inverse_loss, _ = self.icm(
            obs_t, next_obs_t, actions_t)

        # update ICM
        loss = (self.icm_lambda * inverse_loss +
                (1 - self.icm_lambda) * forward_errors.mean())
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        errors = forward_errors.detach().cpu().numpy()
        self._update_stats(errors)

        # normalise
        normalised = errors / (np.sqrt(self.error_var) + 1e-8)
        return normalised # (n_envs,)



In [ ]:
import torch
import torch.nn as nn
import numpy as np
from torch.distributions import Categorical
from stable_baselines3.common.env_util import make_vec_env
from tqdm.notebook import trange
import wandb



class ActorCritic(nn.Module):
    def __init__(self, obs_shape, n_actions):
        super().__init__()

        c, h, w = obs_shape[2], obs_shape[0], obs_shape[1]

        # CNN feature extractor
        self.feature = nn.Sequential(
            nn.Conv2d(c, 32, kernel_size=8, stride=4),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1),
            nn.ReLU(),
            nn.Flatten(),
        )

        # get output format of CNN
        dummy = torch.zeros(1, c, h, w)
        cnn_out_dim = self.feature(dummy).shape[1]

        # head to get logits for futher predictions
        self.head = nn.Sequential(
            nn.Linear(cnn_out_dim, 512),
            nn.ReLU()
        )

        self.actor = nn.Linear(512, n_actions)
        self.critic = nn.Linear(512, 1)

    def _extract(self, x):

        # Input (batch, H, W, C)
        # need (batch, C, H, W) for conv
        if x.ndim == 3:
            x = x.unsqueeze(0)
        x = x.permute(0, 3, 1, 2)
        x = x.float() / 255.0 # normalise to [0, 1]
        return self.head(self.feature(x))

    def forward(self, x):
        features = self._extract(x)
        action = self.actor(features)
        value = self.critic(features)

        return action, value

    def get_action(self, obs):

        logits, value = self.forward(obs)
        dist = Categorical(logits=logits)
        action = dist.sample()
        log_prob = dist.log_prob(action)
        entropy = dist.entropy()
        return action, log_prob, entropy, value.squeeze(-1)

    def evaluate(self, obs, actions):
        logits, value = self.forward(obs)
        dist = Categorical(logits=logits)
        log_prob = dist.log_prob(actions)
        entropy = dist.entropy()
        return log_prob, entropy, value.squeeze(-1)


# Rollout Buffer
class RolloutBuffer:
    def __init__(self, n_steps, n_envs, obs_dim, device):
        self.n_steps = n_steps
        self.n_envs = n_envs
        self.device = device

        self.obs = torch.zeros(n_steps, n_envs, *obs_dim)
        self.actions = torch.zeros(n_steps, n_envs, dtype=torch.long)
        self.log_probs= torch.zeros(n_steps, n_envs)
        self.rewards = torch.zeros(n_steps, n_envs)
        self.values = torch.zeros(n_steps, n_envs)
        self.dones = torch.zeros(n_steps, n_envs)
        self.pos = 0

    def push(self, obs, actions, log_probs, rewards, values, dones):
        self.obs[self.pos] = torch.tensor(obs, dtype=torch.float32)
        self.actions[self.pos] = torch.tensor(actions, dtype=torch.long)
        self.log_probs[self.pos] = torch.tensor(log_probs, dtype=torch.float32)
        self.rewards[self.pos] = torch.tensor(rewards, dtype=torch.float32)
        self.values[self.pos] = torch.tensor(values, dtype=torch.float32)
        self.dones[self.pos] = torch.tensor(dones, dtype=torch.float32)
        self.pos += 1

    def is_full(self):
        return self.pos >= self.n_steps

    def reset(self):
        self.pos = 0
    # Generalised Advantage Estimation
    def compute_gae(self, last_value, gamma=0.99, gae_lambda=0.95):


        advantages = torch.zeros_like(self.rewards)
        last_gae = 0

        for t in reversed(range(self.n_steps)):
          if t == self.n_steps - 1:
              next_value = last_value
          else:
              next_value = self.values[t + 1]

          next_done = self.dones[t]
                                                        #Prevents from leaking advantages from other episodes
          delta = self.rewards[t] + gamma * next_value * (1 - next_done) - self.values[t]
          last_gae = delta + gamma * gae_lambda * (1 - next_done) * last_gae
          advantages[t] = last_gae

        returns = advantages + self.values
        return advantages.to(self.device), returns.to(self.device)

    def get(self):
        """Flatten across steps and envs for minibatch training"""
        obs_shape = self.obs.shape[2:]
        return (
            self.obs.view(-1, *obs_shape).to(self.device),
            self.actions.view(-1).to(self.device),
            self.log_probs.view(-1).to(self.device),
        )



# PPO Algorithm
class PPO:
    def __init__(
        self,
        env,
        learning_rate = 3e-4,
        n_steps = 2048,   # steps per env before each update
        batch_size = 64,
        epochs = 10,     # how many times to reuse each rollout
        gamma = 0.99,
        gae_lambda = 0.95,
        clip_range = 0.2,
        ent_coef = 0.01, # defines how mmuch to explore
        vf_coef = 0.5,
        max_grad_norm = 0.5,
        icm_lr= 1e-4,
        icm_output=256,
        icm_beta=0.5,
        verbose = 1,
        device = "cpu"
    ):
        self.env = env
        self.n_steps = n_steps
        self.batch_size = batch_size
        self.epochs = epochs
        self.gamma = gamma
        self.gae_lambda = gae_lambda
        self.clip_range = clip_range
        self.ent_coef = ent_coef
        self.vf_coef = vf_coef
        self.max_grad_norm = max_grad_norm
        self.verbose = verbose
        self.device = torch.device(device)
        self.n_envs = env.num_envs
        self.early_stop_patience = 10
        self.p_loss_delta_threshold = 1e-3 #
        self.p_loss_history = [] #window of policy losses


        obs_dim = env.observation_space.shape
        n_actions = env.action_space.n

        self.policy = ActorCritic(obs_dim, n_actions).to(self.device)
        self.icm = ICM(
            obs_shape = obs_dim,
            n_actions = n_actions,
            n_envs = self.n_envs,
            feature_dim= icm_output,
            lr = icm_lr,
            beta = icm_beta,
            device = self.device
)
        self.optimizer = torch.optim.Adam(self.policy.parameters(), lr=learning_rate)
        self.buffer = RolloutBuffer(n_steps, self.n_envs, obs_dim, self.device)

    def _collect_rollout(self, obs):
        """Run n_steps in the env and fill the buffer"""
        self.buffer.reset()

        intrinsic_total = []

        for _ in range(self.n_steps):
            obs_t = torch.tensor(obs, dtype=torch.float32).to(self.device)

            with torch.no_grad():
                actions, log_probs, _, values = self.policy.get_action(obs_t)

            new_obs, rewards, dones, _ = self.env.step(actions.cpu().numpy())

            # add intrinsic reward for better exploration, sicne reward in Snake is sparce in early stages
            # and maybe will prevent from early convergence
            if self.icm is not None:
                intrinsic = self.icm.compute_intrinsic(
                    obs, new_obs, actions.cpu().numpy())
                intrinsic = np.clip(intrinsic, 0, 1.0)  # safeguard after normalisation

                rewards = rewards + self.icm.beta * intrinsic
                intrinsic_total.append(intrinsic.mean())

            self.buffer.push(obs, actions.cpu().numpy(),
                             log_probs.cpu().numpy(), rewards,
                             values.cpu().numpy(), dones)
            obs = new_obs

        # Bootstrap value for last state
        with torch.no_grad():
            obs_t = torch.tensor(obs, dtype=torch.float32).to(self.device)
            _, last_value = self.policy(obs_t)
            last_value = last_value.squeeze(-1).cpu()

        return obs, last_value, np.mean(intrinsic_total)

    def _update_icm(self, obs, next_obs, actions):
      total_size = obs.shape[0]
      for _ in range(self.epochs):
          indices = torch.randperm(total_size)
          for start in range(0, total_size, self.batch_size):
              idx = indices[start:start + self.batch_size]
              self.icm.update(obs[idx], next_obs[idx], actions[idx])

    def _update(self, advantages, returns):
        """Run epochs of minibatch updates on the collected rollout"""
        obs, actions, old_log_probs = self.buffer.get()
        advantages_flat = advantages.view(-1)
        returns_flat = returns.view(-1)

        # Normalise advantages — reduces variance, stabilises training
        advantages_flat = (advantages_flat - advantages_flat.mean()) / (advantages_flat.std() + 1e-8)

        total_size = obs.shape[0]
        policy_losses, value_losses, entropy_losses = [], [], []

        for _ in range(self.epochs):
            # Shuffle for minibatches
            indices = torch.randperm(total_size)

            for start in range(0, total_size, self.batch_size):
                idx = indices[start:start + self.batch_size]

                mb_obs = obs[idx]
                mb_actions = actions[idx]
                mb_old_lp = old_log_probs[idx]
                mb_advantages = advantages_flat[idx]
                mb_returns = returns_flat[idx]

                # Evaluate current policy on old actions
                new_log_probs, entropy, values = self.policy.evaluate(mb_obs, mb_actions)
                ratio = torch.exp(new_log_probs - mb_old_lp)

                # Clipped surrogate objective
                surr1 = ratio * mb_advantages
                surr2 = torch.clamp(ratio, 1 - self.clip_range,
                                           1 + self.clip_range) * mb_advantages
                policy_loss = -torch.min(surr1, surr2).mean()

                # Value loss
                value_loss = nn.functional.mse_loss(values, mb_returns)

                # acts as epsilon greedy in DQN to encourage exploration
                entropy_loss = -entropy.mean()

                # Combined loss
                loss = policy_loss \
                    + self.vf_coef * value_loss \
                    + self.ent_coef * entropy_loss

                self.optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.policy.parameters(), self.max_grad_norm)
                self.optimizer.step()

                policy_losses.append(policy_loss.item())
                value_losses.append(value_loss.item())
                entropy_losses.append(entropy_loss.item())
        #for logging
        return (np.mean(policy_losses),
                np.mean(value_losses),
                np.mean(entropy_losses))



    def _is_policy_stuck(self, p_loss):
        """
        Returns True if policy loss has been flat for too long.
        Flat = std of recent p_loss window < threshold
        """
        self.p_loss_history.append(p_loss)

        window = 5  # look at last 5 updates
        if len(self.p_loss_history) < window:
            return False

        recent = self.p_loss_history[-window:]
        delta  = np.std(recent)  # how much p_loss is varying

        is_stuck = delta < self.p_loss_delta_threshold
        return is_stuck



    def learn(self, n_updates, eval_env=None, eval_every=10, eval_episodes=5):
        total_timesteps = n_updates * self.n_steps * self.n_envs
        obs = self.env.reset()
        num_timesteps = 0
        update = 0
        stuck_counter = 0
        best_reward = -np.inf

        with trange(total_timesteps, desc="PPO Training", unit="step") as pbar:
            while num_timesteps < total_timesteps:

                obs, last_value, intrinsic = self._collect_rollout(obs)
                num_timesteps  += self.n_steps * self.n_envs
                update += 1
                pbar.update(self.n_steps * self.n_envs)

                advantages, returns = self.buffer.compute_gae(
                    last_value, self.gamma, self.gae_lambda)

                p_loss, v_loss, e_loss = self._update(advantages, returns)

                log = {
                    "policy_loss": p_loss,
                    "value_loss": v_loss,
                    "entropy_loss": e_loss,
                    "intrinsic": intrinsic
                }

                if eval_env is not None and update % eval_every == 0:
                    self.policy.eval()
                    mean_r, std_r = evaluate_learning(self, eval_env, n_episodes=eval_episodes)
                    self.policy.train()

                    # Track best model
                    if mean_r > best_reward:
                        best_reward = mean_r
                        torch.save(self.policy.state_dict(), "best_ppo.pt")

                    # check if policy loss is stuck
                    if self._is_policy_stuck(p_loss):
                        stuck_counter += 1
                        print(f"\n Policy loss flat "
                            f"({stuck_counter}/{self.early_stop_patience}) "
                            f"| p_loss_std={np.std(self.p_loss_history[-5:]):.6f} "
                            f"| p_loss={p_loss:.4f} "
                            f"| mean_r={mean_r:.2f}")

                        if stuck_counter >= self.early_stop_patience:
                            print(f"\nEarly stopping: policy loss flat for "
                                f"{self.early_stop_patience} consecutive evals. "
                                f"Best reward: {best_reward:.2f}")
                            wandb.log({**log, "early_stop": 1})
                            break
                    else:
                        stuck_counter = 0  # reset if p_loss starts moving again

                    log["mean_reward"] = mean_r
                    log["std_reward"] = std_r
                    log["stuck_counter"] = stuck_counter
                    log["best_reward"] = best_reward

                    pbar.set_postfix({
                        "p_loss": f"{p_loss:.4f}",
                        "v_loss": f"{v_loss:.4f}",
                        "entropy": f"{e_loss:.4f}",
                        "intrinsic":f"{intrinsic}",
                        "mean_r": f"{mean_r:.2f}",
                        "std_r": f"{std_r:.2f}",
                        "collapse": f"{stuck_counter}/{self.early_stop_patience}"
                    })
                else:
                    pbar.set_postfix({
                        "p_loss": f"{p_loss:.4f}",
                        "v_loss": f"{v_loss:.4f}",
                        "entropy": f"{e_loss:.4f}",
                        "intrinsic":f"{intrinsic}",
                    })

                wandb.log(log)

        return self



def evaluate_learning(model, env, n_episodes=5, deterministic=True):
    device = model.device
    n_envs = len(env.envs)

    episode_rewards = []
    current_rewards = np.zeros(n_envs)
    episodes_done = 0

    obs = env.reset()

    while episodes_done < n_episodes:
        obs_t = torch.tensor(obs, dtype=torch.float32).to(device)
        with torch.no_grad():
            logits, _ = model.policy(obs_t)

        if deterministic:
            actions = torch.argmax(logits, dim=-1).cpu().numpy()
        else:
            actions = Categorical(logits=logits).sample().cpu().numpy()

        obs, rewards, dones, _ = env.step(actions)
        current_rewards += rewards

        for i, done in enumerate(dones):
            if done and episodes_done < n_episodes:
                episode_rewards.append(current_rewards[i])
                current_rewards[i] = 0.0
                episodes_done += 1

    return np.mean(episode_rewards), np.std(episode_rewards)

In [ ]:
import gymnasium as gym
import numpy as np

class RewardShapingWrapper(gym.Wrapper):
    def __init__(self, env,
                 step_penalty = -0.01,
                 hunger_cap = 36,
                 death_penalty = -1.0,   # hit wall or body
): # moving towards food
        super().__init__(env)
        self.step_penalty = step_penalty
        self.hunger_cap = hunger_cap
        self.death_penalty = death_penalty
        self.steps_since_food = 0


    def reset(self, **kwargs):
        self.steps_since_food = 0
        obs, info = self.env.reset(**kwargs)
        return obs, info


    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)

        # Death penalty
        if terminated and info.get("message") in ("hit the wall",
                                                   "Come into contact with the body"):
            reward += self.death_penalty

        # Hunger / starvation
        if reward >= 1:
            self.steps_since_food = 0
        else:
            self.steps_since_food += 1

        if self.steps_since_food >= self.hunger_cap:
            terminated = True
            info["message"] = "Starved"
            self.steps_since_food = 0
            reward += self.death_penalty   # same penalty as dying

        # Direction reward
        # Only shape direction when alive and no food eaten this step
        # if not terminated and reward < 1:
        #     curr_distance = self._get_distance()

            # if self.prev_distance is not None:
            #     if curr_distance < self.prev_distance:
            #         reward += self.direction_reward # closer -> reward
            #     elif curr_distance > self.prev_distance:
            #         reward -= self.direction_reward # further -> penalty

            # self.prev_distance = curr_distance
        # else:
        #     # Reset distance tracking after food eaten or death
        #     self.prev_distance = self._get_distance() if not terminated else None

        # Step penalty
        reward += self.step_penalty

        return obs, reward, terminated, truncated, info

In [ ]:
import numpy as np
import torch
from stable_baselines3.common.env_util import make_vec_env

def test_run():
    DEVICE = torch.device( "cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {DEVICE}")
    run = wandb.init()

    # NEVER EVER render mp4 here. It makes the whole training 4 times slower and GPU stalls
    # Maybe if use stablebaseline renderer then will be better
    vec_env = make_vec_env(
    "tcs-v2",
    n_envs=16,
    env_kwargs={
        "height": 8,
        "width": 8,
        "obs_type": "rgb",
        "image_size": 10,
    },
    wrapper_class=RewardShapingWrapper,
    wrapper_kwargs={
        "step_penalty": -0.01
    }
)
    eval_env = make_vec_env(
        "tcs-v2",
        n_envs=5,
        env_kwargs={
            "height": 8,
            "width": 8,
            "obs_type": "rgb",
            "image_size": 10
        },

    )

    model = PPO(
        env=vec_env,
        learning_rate=1.5e-4,
        batch_size=1024,
        epochs=4,
        ent_coef=0.1,
        vf_coef=.25,
        n_steps=4096,
        device="cuda",

    )


    model.learn(n_updates=100, eval_env=eval_env, eval_every=5)

    torch.save(model.policy.state_dict(), "ppo_policy.pt")
    print("Saved to ppo_policy.pt")

# test_run()

In [ ]:
import wandb
from stable_baselines3.common.env_util import make_vec_env

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

best_reward = -np.inf

def train():
    global best_reward
    run = wandb.init()
    config = run.config

    vec_env = make_vec_env(
        "tcs-v2",
        n_envs=16,
        env_kwargs={
            "height": 8,
            "width": 8,
            "obs_type": "rgb",
            "image_size": 10
        },
        wrapper_class=RewardShapingWrapper,
        wrapper_kwargs={
            "step_penalty": config.step_penalty,
            "hunger_cap": config.hunger_cap,
            "death_penalty": config.death_penalty,
        }
    )

    eval_env = make_vec_env(
        "tcs-v2",
        n_envs=4,
        env_kwargs={
            "height": 8,
            "width": 8,
            "obs_type": "rgb",
            "image_size": 10
        }
    )

    model = PPO(
        env = vec_env,
        learning_rate = config.learning_rate,
        n_steps = config.n_steps,
        batch_size = config.batch_size,
        epochs = config.epochs,
        gamma = config.gamma,
        gae_lambda = config.gae_lambda,
        clip_range = config.clip_range,
        ent_coef = config.ent_coef,
        vf_coef = config.vf_coef,
        icm_lr = config.icm_lr,
        icm_output = config.icm_output,
        icm_beta = config.icm_beta,
        device = DEVICE
    )

    model.learn(n_updates=100, eval_env=eval_env)

    mean_reward, std_reward = evaluate_learning(model, eval_env, n_episodes=50)
    wandb.log({
        "mean_reward": mean_reward,
        "std_reward": std_reward,
    })

    if mean_reward > best_reward:
        best_reward = mean_reward
        model_path = "best_sweep_model.pt"
        torch.save({
            "policy": model.policy.state_dict(),
            "mean_reward": best_reward,
            "config": dict(config),
        }, model_path)
        wandb.save(model_path)
        run.summary["best_reward"] = best_reward
        print(f"New best: {best_reward:.2f} | Run: {run.id}")

    eval_env.close()
    vec_env.close()
    run.finish()


In [ ]:
# Bayes search set up with wandb

sweep_config = {
    "method": "bayes",
    "metric": {
        "name": "mean_reward",
        "goal": "maximize"
    },
    "parameters": {
        "learning_rate": {
            "distribution": "log_uniform_values",
            "min": 1e-5,
            "max": 5e-4
        },
        "gamma": {
            "values": [0.99, 0.995, 0.999, 0.9995]
        },
        "gae_lambda": {
            "values": [0.90, 0.95, 0.98]
        },
        "clip_range": {
            "values": [0.1, 0.2, 0.3]
        },
        "ent_coef": {
            "distribution": "log_uniform_values",
            "min": 1e-3,
            "max": 0.15
        },
        "vf_coef": {
            "values": [0.5, 1.0, 1.5, 2.0]
        },
        "n_steps": {
            "values": [1024, 2048, 3072, 4096]
        },
        "batch_size": {
            "values": [512, 768, 1024]
        },
        "epochs": {
            "values": [4, 6, 8]
        },
        # reward shaping
        "step_penalty": {
            "distribution": "uniform",
            "min": -0.01,
            "max": -0.001
        },
        "hunger_cap": {
            "values": [24, 36, 48, ]
        },
        "death_penalty": {
            "distribution": "uniform",
            "min": -3.0,
            "max": -0.5
        },
        "direction_reward": {
            "distribution": "uniform",
            "min": 0.0,
            "max": 0.2
        },
        # icm
        "icm_beta": {
            "distribution": "uniform",
            "min": 0.1,
            "max": 1.0
        },
        "icm_lr": {
            "distribution": "log_uniform_values",
            "min": 1e-5,
            "max": 1e-3
        },
        "icm_output": {
            "values": [128, 256, 512]
        }
    }
}

sweep_id = wandb.sweep(sweep_config, project="Snake_PPO")
wandb.agent(sweep_id, function=train, count=20)

checkpoint = torch.load("best_sweep_model.pt", weights_only=False)
print(f"Best reward: {checkpoint['mean_reward']:.2f}")
print(f"Best config: {checkpoint['config']}")


In [ ]:
#Test a standalone policy.
def test_policy(policy, env, n_episodes=5, deterministic=True, render=False):

    policy.eval()
    device = next(policy.parameters()).device
    episode_rewards = []

    for ep in range(n_episodes):
        obs = env.reset()
        done = [False] * env.num_envs
        total = np.zeros(env.num_envs)

        while not all(done):
            obs_t = torch.tensor(obs, dtype=torch.float32).to(device)

            with torch.no_grad():
                logits, _ = policy(obs_t)

            if deterministic:
                actions = torch.argmax(logits, dim=-1).cpu().numpy()
            else:
                actions = Categorical(logits=logits).sample().cpu().numpy()

            obs, rewards, done, _ = env.step(actions)
            total += np.array(rewards)

        episode_rewards.append(total.mean())

        if render:
            print(f"Episode {ep + 1:>3} | Reward: {total.mean():.2f}")

    mean_r = np.mean(episode_rewards)
    std_r = np.std(episode_rewards)
    print(f"\nResult over {n_episodes} episodes | Mean: {mean_r:.2f} ± {std_r:.2f}")
    return mean_r, std_r

eval_env = make_vec_env(
    "tcs-v2",
    n_envs=1,
    env_kwargs={
        "height": 8,
        "width": 8,
        "obs_type": "rgb",
        "image_size": 10,
    },

)

In [ ]:
checkpoint = torch.load("ppo_policy.pt", weights_only=False)

obs_dim = eval_env.observation_space.shape
n_actions = eval_env.action_space.n

policy = ActorCritic(obs_dim, n_actions)
policy.load_state_dict(checkpoint)

mean, std = test_policy(policy, eval_env, n_episodes=1000, render=True)

print(f"Mean: {mean} | Std: {std}")